# Gazebo litter pilot fine-tuning
Runtime > Change runtime type で GPU を選択してから実行します。元の重みは読み取り入力として扱い、Drive上の別ディレクトリへ結果を保存します。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/robot-1-colab'
BUNDLE = f'{DRIVE_ROOT}/gazebo_litter_pilot_colab.zip'
RUN_ROOT = f'{DRIVE_ROOT}/gazebo_litter_runs'

In [ ]:
!nvidia-smi
!pip -q install 'ultralytics>=8.4,<9'
import torch
assert torch.cuda.is_available(), 'GPU runtime is not enabled'
print(torch.cuda.get_device_name(0))

In [ ]:
import shutil, yaml
from pathlib import Path
WORK = Path('/content/gazebo_litter_pilot_work')
if WORK.exists(): shutil.rmtree(WORK)
WORK.mkdir()
shutil.unpack_archive(BUNDLE, WORK)
DATASET = WORK / 'gazebo_litter_pilot'
data = yaml.safe_load((DATASET / 'data.yaml').read_text())
data['path'] = str(DATASET)
(DATASET / 'data.colab.yaml').write_text(yaml.safe_dump(data, sort_keys=False))
ORIGINAL = WORK / 'weights/original_best.pt'
assert ORIGINAL.exists() and (DATASET / 'manifest.json').exists()

In [ ]:
# 新規学習。OOM時だけ batch=8 -> 4 -> 2 と下げます。
from ultralytics import YOLO
import gc, os
os.makedirs(RUN_ROOT, exist_ok=True)
train_result = None
for batch in (8, 4, 2):
    try:
        model = YOLO(str(ORIGINAL))
        train_result = model.train(data=str(DATASET/'data.colab.yaml'), imgsz=640, batch=batch, epochs=50, patience=10, seed=42, deterministic=True, device=0, project=RUN_ROOT, name='gazebo_litter_pilot', exist_ok=True, save=True, save_period=1, workers=2)
        break
    except RuntimeError as exc:
        if 'out of memory' not in str(exc).lower() or batch == 2: raise
        print(f'CUDA OOM at batch={batch}; retrying with smaller batch')
        del model; gc.collect(); torch.cuda.empty_cache()
assert train_result is not None

## 切断後の再開
下のセルだけを使う場合も、Drive mount、install、展開セルを先に実行してください。`last.pt`は各epochでDriveへ保存されます。

In [ ]:
# 必要な場合のみコメント解除して実行
# from ultralytics import YOLO
# LAST = Path(RUN_ROOT) / 'gazebo_litter_pilot/weights/last.pt'
# assert LAST.exists()
# YOLO(str(LAST)).train(resume=True, device=0)

In [ ]:
# 学習曲線と重みはRUN_ROOTに直接保存済み。固定test holdoutで旧新を比較します。
RUN = Path(RUN_ROOT) / 'gazebo_litter_pilot'
FINETUNED = RUN / 'weights/best.pt'
assert FINETUNED.exists()
!python {WORK/'tools/evaluate_yolo_holdout.py'} --dataset {DATASET} --original {ORIGINAL} --finetuned {FINETUNED} --output {RUN/'holdout_evaluation'} --device 0
from IPython.display import display, Image
display(Image(filename=str(RUN/'results.png')))
print((RUN/'holdout_evaluation/COMPARISON.md').read_text())

In [ ]:
# Macへ戻す成果物を1つにまとめます。元best.ptは変更されません。
ARCHIVE = shutil.make_archive(str(Path(DRIVE_ROOT)/'gazebo_litter_pilot_result'), 'zip', RUN)
print('Download from Drive:', ARCHIVE)